In [1]:
from pathlib import Path
import sys
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from datasets import load_dataset as load_hf_dataset
import numpy as np
import torch
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "MOC" / "utilities").exists()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from MOC.utilities.LogicNet import LogicNet
from MOC.utilities.train_model import train_model

# Create and use the model

transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

/home/maciek/Projects/ConfLGN/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = LogicNet(num_classes=10,dense_num=3,base_dense_dims=[1280,640,320], conv_num=3, kernel_multiplier=3, k=16)






cipadupa2115




cipadupa2115




cipadupa2115













In [ ]:
sigma_values = np.arange(0.5, 3.0 + 0.001, 0.5)

experiment_configs = [
    {
        "name": f"gaussian_sigma_{sigma:.1f}",
        "model_kwargs": {
            "gauss_sigma": float(sigma),
            "connection_init_method": "gaussian",
        },
    }
    for sigma in sigma_values
]

experiment_configs.append(
    {
        "name": "random_unique",
        "model_kwargs": {
            "gauss_sigma": 1.5,  # ignored by random-unique conv init, but harmless
            "connection_init_method": "random-unique",
        },
    }
)

results = {}

for config in experiment_configs:
    print(f"\n=== Running {config['name']} ===")

    torch.manual_seed(0)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(0)

    model = LogicNet(
        num_classes=10,  # use 2 for PatchCamelyon
        dense_num=3,
        base_dense_dims=[1280, 640, 320],
        conv_num=3,
        kernel_multiplier=3,
        k=16,
        **config["model_kwargs"],
    )

    model, history = train_model(
        model,
        train_dataset,
        test_dataset,
        lr=2e-1,
        weight_decay=0,
        batch_size=512,
        num_iterations=2500,
        metrics_every=100,
        force_cpu=False,
    )

    results[config["name"]] = {
        "history": history,
        "final_acc_discrete": history["test_acc_discrete"][-1],
        "final_acc_relaxed": history["test_acc_relaxed"][-1],
    }

cuda



KeyboardInterrupt: 